# T2 step 5: the reports read the system's own damage facts (C4), for two satellite photo pairs

Roadmap T2 step 5, built 20 Nov 2026 (Abrar took step 5 over, so Ridita can work on Layer 3). A copy of
`t2_disasteller_local_run.ipynb` (Ridita's, unchanged): the same model server, installs and pass rules, with the
flow `fixed_flow_c4.py` instead of `fixed_flow.py`.

**What changes for the reports.** They describe one before-and-after satellite pair, not July's six ground photos:
- the 7B describes the before- and the after-photo (the photo tool, as before; no sticker question);
- every report's facts start with C4's facts for the photo, read through the bridge `disasteller_inputs.py`: the
  counts by damage state (on xBD photos, counts of *areas*, since a row there can hold several touching
  buildings), the hot spots and the safe areas, all called the model's estimates;
- the damage grades are written by the code from the counts (destroyed = EMS-98 Grade 5, damaged = Grades 2-4,
  intact = at most Grade 1, each citing its EMS-98 section), never by the language model;
- the map reading is written by the code from the hot spots, and the labelled map is the after-photo with the hot
  spots (H1...) and safe areas (S1...) drawn on it.

**The two scenes** (rule fixed before any report: of the clean test pairs, the earthquake pair from the sharper
photos and the xBD pair with the most truly destroyed pixels): `turkey_earthquake4_post_327.png` (one row per
building) and `santa_rosa_wildfire_00000155_post_disaster.png` (patches). The flow runs once per scene: six reports
each.

**Inputs (+ Add Input):**
1. the `offline-rag-kr2` dataset (the guideline search index);
2. the `disasteller-offline-c4` dataset: upload `kaggle_upload/disasteller-offline-c4.zip` (made by
   `make_t2_step5_upload.py`) as a private dataset with this name;
3. the output of `t2_prep_packages.ipynb` (installer files);
4. the output of `t2_prep_model.ipynb` (the model);
5. the benchmark mirror `redwannewaz/disasterm3-bench-mirror-v2` (the two pairs' photos).

**Settings:** Accelerator **GPU T4 x2**, Internet **off**, then Save Version → **Save & Run All (Commit)**. About
45-60 min: loading the 7B, then two scenes of about 15-20 min each.

**Expectation, fixed before the run:** both scenes pass the five pass rules (`check_run.py`); every report states
the counts exactly as the facts give them; the xBD scene's reports speak of areas, not buildings, for its counts.
Step 7 then checks every number in the reports against the table.

**Outputs** (in `t2_step5/`, also zipped as `t2_step5_c4_internet_off.zip`): per scene, `run_<scene>/` with the six
reports and the labelled map (`reports/offline/`), `netlog.json`, `run_log.txt`, `llm_calls.jsonl` and
`t2_run_summary.json`; and `vllm_server.log` and `t2_notebook_info.json`.

In [1]:
import glob, json, os, shutil, socket, subprocess, sys, threading, time, urllib.request, zipfile

FLOW = "c4"                     # fixed_flow_c4.py: the fixed flow on a satellite pair, with C4's damage facts
SERVED_NAME = "disasterm3-7b"   # the model's name on the server = LOCAL_LLM_MODEL
MAX_MODEL_LEN = 32768           # tokens, question + answer = LOCAL_LLM_CONTEXT (16384 in practice run 2: too small)
PORT = 8000
PACKAGES = ["vllm==0.25.1", "crewai==1.15.2"]   # the same as in t2_prep_packages
RUN_TIMEOUT_HOURS = 2           # per scene: stop the flow if it runs longer (for example, a model in a loop)
WORK = "/kaggle/working/t2_step5"
os.makedirs(WORK, exist_ok=True)
T0 = time.time()


def minutes():
    return f"{(time.time() - T0) / 60:5.1f} min"

In [2]:
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    INTERNET = "on"
    print("Internet is ON: this is the practice run. For the proof, turn it off and run again.")
except OSError:
    INTERNET = "off"
    print("Internet is OFF: nothing below can download anything. This is the proof run.")

Internet is OFF: nothing below can download anything. This is the proof run.


Find the four inputs. If Kaggle kept an uploaded zip as a zip, it is unpacked into `/kaggle/working/unzipped/`.

In [3]:
def find_dir(marker, what, zip_name=None):
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True))
    if not hits and zip_name:
        for z in glob.glob(f"/kaggle/input/**/{zip_name}", recursive=True):
            with zipfile.ZipFile(z) as f:
                f.extractall(f"/kaggle/working/unzipped/{os.path.basename(z)[:-4]}")
        hits = sorted(glob.glob(f"/kaggle/working/unzipped/**/{marker}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{what} is missing. Add it with + Add Input.")
    return os.path.dirname(hits[0])


RAG_DIR = find_dir("guideline_index.py", "The offline-rag-kr2 dataset", "offline-rag-kr2*.zip")
DT_INPUT = find_dir("fixed_flow_c4.py", "The disasteller-offline-c4 dataset (make_t2_step5_upload.py)",
                    "disasteller-offline-c4*.zip")
MODEL_DIR = find_dir("t2_model_source.json", "The output of t2_prep_model.ipynb")
WHEELS_DIR = find_dir("t2_wheels_manifest.json", "The output of t2_prep_packages.ipynb")
for name in ("RAG_DIR", "DT_INPUT", "MODEL_DIR", "WHEELS_DIR"):
    print(f"{name:10} {globals()[name]}")

code = json.load(open(f"{DT_INPUT}/disasteller_manifest.json"))
model_source = json.load(open(f"{MODEL_DIR}/t2_model_source.json"))
print("\nDisasTeller code: packed", code["made"], "| git commit", code["git_commit"],
      "| uncommitted changes:", code["uncommitted_changes_in_disasteller_offline"])
print("Model:", model_source["repo"], "| commit", model_source["commit"])

RAG_DIR    /kaggle/input/datasets/redwannewaz/offline-rag-kr2
DT_INPUT   /kaggle/input/datasets/redwannewaz/disasteller-offline-c4
MODEL_DIR  /kaggle/input/notebooks/redwannewaz/t2-prep-model/model
WHEELS_DIR /kaggle/input/notebooks/redwannewaz/t2-prep-packages/wheels

DisasTeller code: packed 2026-11-20 07:57:14 | git commit 1a5f38094 | uncommitted changes: True
Model: AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP | commit def8f6c81889ffb68904afdc90c3a1389697d73b


Put the DisasTeller files where its code expects them, before anything uses the GPU.
Kaggle can change the folder layout of an uploaded dataset: in practice run 1 (3 Nov), the code was
there but `Disaster_image/Local_image/Picture1.png` was not. So each file listed in
`disasteller_manifest.json` is found by its name, checked against its sha256, and copied to
`/kaggle/working/disasteller/` at the path the manifest gives. DisasTeller then runs from that copy.

In [4]:
import hashlib


def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


DT_DIR = "/kaggle/working/disasteller"
shutil.rmtree(DT_DIR, ignore_errors=True)
FOUND_AT, missing = {}, []
for rel, digest in sorted(code["sha256"].items()):
    name = os.path.basename(rel)
    candidates = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)
                        + glob.glob(f"/kaggle/working/unzipped/**/{name}", recursive=True))
    match = next((c for c in candidates if sha256(c) == digest), None)
    if match is None:
        missing.append(f"{rel} (files with this name: {candidates or 'none'})")
        continue
    dest = os.path.join(DT_DIR, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(match, dest)
    FOUND_AT[rel] = match

for rel, path in FOUND_AT.items():
    print(f"{rel:45} <- {path}")
if missing:
    raise FileNotFoundError("Missing or changed in the disasteller-offline dataset: " + "; ".join(missing)
                            + ". Upload disasteller-offline.zip again, as a new version of the dataset.")
shutil.copy2(f"{DT_INPUT}/disasteller_manifest.json", DT_DIR)
print(f"\nAll {len(FOUND_AT)} files are in {DT_DIR}, each with the sha256 in the manifest.")

# The two scenes (from the checked copy, whatever layout Kaggle gave the dataset) and their photos, found by name
# in the benchmark mirror
SCENES = json.load(open(os.path.join(DT_DIR, "c4_scenes", "scenes.json"), encoding="utf-8"))
PHOTOS = {}
for scene, info in SCENES.items():
    for role, name in (("before", info["before_image"]), ("after", scene)):
        hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
        if not hits:
            raise FileNotFoundError(f"{name} is missing: add the benchmark mirror (redwannewaz/disasterm3-bench-mirror-v2).")
        PHOTOS[(scene, role)] = hits[0]
for (scene, role), path in PHOTOS.items():
    print(f"{role:6} {path}")

# This session's copy of run_with_netlog.py also knows the flow "c4" (the dataset's file and the repository's stay as
# they are): the network log then records fixed_flow_c4.py as the main script.
NETLOG = os.path.join(DT_DIR, "run_with_netlog.py")
netlog_text = open(NETLOG, encoding="utf-8").read()
OLD_FLOWS = '"fixed": "fixed_flow.py"}'
assert netlog_text.count(OLD_FLOWS) == 1, "run_with_netlog.py changed: check its FLOWS before adding c4"
with open(NETLOG, "w", encoding="utf-8") as f:
    f.write(netlog_text.replace(OLD_FLOWS, '"fixed": "fixed_flow.py", "c4": "fixed_flow_c4.py"}'))
print("run_with_netlog.py (this session's copy) knows the flow 'c4': fixed_flow_c4.py")

DisasterManagement_teamwork_simulation.py     <- /kaggle/input/datasets/redwannewaz/disasteller-offline-c4/DisasterManagement_teamwork_simulation.py
Disaster_image/Global_image/2.jpg             <- /kaggle/input/datasets/redwannewaz/disasteller-offline-c4/Disaster_image/Global_image/2.jpg
Disaster_image/Local_image/Picture1.png       <- /kaggle/input/datasets/redwannewaz/disasteller-offline-c4/Disaster_image/Local_image/Picture1.png
Disaster_image/Local_image/Picture2.png       <- /kaggle/input/datasets/redwannewaz/disasteller-offline-c4/Disaster_image/Local_image/Picture2.png
Disaster_image/Local_image/Picture3.png       <- /kaggle/input/datasets/redwannewaz/disasteller-offline-c4/Disaster_image/Local_image/Picture3.png
Disaster_image/Local_image/Picture4.png       <- /kaggle/input/datasets/redwannewaz/disasteller-offline-c4/Disaster_image/Local_image/Picture4.png
Disaster_image/Local_image/Picture5.png       <- /kaggle/input/datasets/redwannewaz/disasteller-offline-c4/Disaster_image/

In [5]:
try:
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:
    gpus = []
print(gpus)
assert len(gpus) >= 2, "Set Accelerator: GPU T4 x2. The 7B is split across two GPUs."

['Tesla T4, 15360 MiB', 'Tesla T4, 15360 MiB']


Install vLLM and CrewAI **from the installer files only** (`--no-index`: pip never looks online).

In [6]:
t = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", WHEELS_DIR, *PACKAGES],
                   capture_output=True, text=True)
with open(f"{WORK}/pip_install_log.txt", "w") as f:
    f.write(r.stdout + "\n" + r.stderr)
print(r.stdout[-1500:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("pip install failed: see t2_run/pip_install_log.txt")

VERSIONS = json.loads(subprocess.run([sys.executable, "-c", (
    "import json, importlib.metadata as m\n"
    "v = {}\n"
    "for p in ('vllm', 'crewai', 'torch', 'transformers', 'openai', 'numpy', 'pillow'):\n"
    "    try: v[p] = m.version(p)\n"
    "    except Exception: v[p] = None\n"
    "print(json.dumps(v))")], capture_output=True, text=True).stdout)
print(f"Installed in {(time.time() - t) / 60:.1f} min:", VERSIONS)

.3 lance-namespace-0.13.0 lance-namespace-urllib3-client-0.13.0 lancedb-0.30.0 lark-1.2.2 llguidance-1.7.6 llvmlite-0.47.0 lm-format-enforcer-0.11.3 loguru-0.7.3 mcp-1.26.0 mistral_common-1.13.1 model-hosting-container-standards-0.1.16 msgspec-0.22.0 numba-0.65.0 nvidia-cuda-nvcc-13.2.86 nvidia-cuda-tileiras-13.2.86 nvidia-cudnn-frontend-1.28.0 nvidia-cutlass-dsl-4.5.2 nvidia-cutlass-dsl-libs-base-4.5.2 nvidia-cutlass-dsl-libs-cu13-4.5.2 nvidia-nvvm-13.2.86 nvtx-0.2.15 onnxruntime-1.31.1 openai-harmony-0.0.8 opentelemetry-exporter-otlp-1.42.1 opentelemetry-exporter-otlp-proto-common-1.42.1 opentelemetry-exporter-otlp-proto-grpc-1.42.1 opentelemetry-exporter-otlp-proto-http-1.42.1 opentelemetry-proto-1.42.1 opentelemetry-semantic-conventions-ai-0.5.1 outlines_core-0.2.14 partial-json-parser-0.2.1.1.post7 pdfminer.six-20260107 pdfplumber-0.11.10 pendulum-3.3.0 portalocker-2.7.0 posthog-5.4.0 prometheus-fastapi-instrumentator-8.1.0 protobuf-6.33.6 pybase64-1.5.1 pycountry-26.2.16 pydantic

The model's chat template (how a conversation is laid out for the model). If the model folder has
none, Qwen2.5-VL's standard template is used. It has no section for tools, which is why
DisasTeller uses CrewAI's text format for tool calls (`LOCAL_LLM_NATIVE_TOOLS=0`).

In [7]:
QWEN25_VL_TEMPLATE = "{% set image_count = namespace(value=0) %}{% set video_count = namespace(value=0) %}{% for message in messages %}{% if loop.first and message['role'] != 'system' %}<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n{% endif %}<|im_start|>{{ message['role'] }}\n{% if message['content'] is string %}{{ message['content'] }}<|im_end|>\n{% else %}{% for content in message['content'] %}{% if content['type'] == 'image' or 'image' in content or 'image_url' in content %}{% set image_count.value = image_count.value + 1 %}{% if add_vision_id %}Picture {{ image_count.value }}: {% endif %}<|vision_start|><|image_pad|><|vision_end|>{% elif content['type'] == 'video' or 'video' in content %}{% set video_count.value = video_count.value + 1 %}{% if add_vision_id %}Video {{ video_count.value }}: {% endif %}<|vision_start|><|video_pad|><|vision_end|>{% elif 'text' in content %}{{ content['text'] }}{% endif %}{% endfor %}<|im_end|>\n{% endif %}{% endfor %}{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"

config = json.load(open(f"{MODEL_DIR}/config.json"))
print("Architecture:", config.get("architectures"), "| dtype:", config.get("torch_dtype") or config.get("dtype"))
CHAT_TEMPLATE_ARGS = []
if model_source.get("chat_template_in"):
    print("Chat template: the model's own, in", model_source["chat_template_in"])
else:
    path = f"{WORK}/qwen2_5_vl_chat_template.jinja"
    with open(path, "w") as f:
        f.write(QWEN25_VL_TEMPLATE)
    CHAT_TEMPLATE_ARGS = ["--chat-template", path]
    print("Chat template: Qwen2.5-VL's standard one, written to", path)

Architecture: ['Qwen2_5_VLForConditionalGeneration'] | dtype: float16
Chat template: the model's own, in ['chat_template.jinja']


Start the vLLM server on this machine only (`127.0.0.1`), split across the two T4s, in half
precision, with its usage report off. It allows one picture per request, and pictures up to
1344 × 1344 pixels (the map is 1080 × 1522). Loading the 7B takes several minutes.

In [8]:
vllm_cli = shutil.which("vllm")
cmd = ([vllm_cli, "serve", MODEL_DIR] if vllm_cli
       else [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL_DIR])
cmd += ["--served-model-name", SERVED_NAME, "--host", "127.0.0.1", "--port", str(PORT),
        "--tensor-parallel-size", "2", "--dtype", "half", "--enforce-eager",
        "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92", "--max-num-seqs", "8",
        "--limit-mm-per-prompt", json.dumps({"image": 1}),
        "--mm-processor-kwargs", json.dumps({"max_pixels": 1344 * 1344}),
        *CHAT_TEMPLATE_ARGS]
server_env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                  VLLM_NO_USAGE_STATS="1", DO_NOT_TRACK="1")
print(" ".join(cmd))
server_log = open(f"{WORK}/vllm_server.log", "w")
server = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT, env=server_env)

ready = False
for _ in range(180):  # up to 30 minutes
    if server.poll() is not None:
        break
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
        ready = True
        break
    except OSError:
        time.sleep(10)

log_text = open(f"{WORK}/vllm_server.log", errors="replace").read()
if not ready:
    print(log_text[-5000:])
    raise RuntimeError(
        "vLLM did not start: its log is above and in t2_run/vllm_server.log. If it says the KV cache "
        "can't hold max_model_len tokens, set MAX_MODEL_LEN = 16384 in the first cell and run again "
        "(in practice run 2 the KV cache held 198,176 tokens, so 32768 should fit).")
print(f"[{minutes()}] vLLM is ready")
for line in log_text.splitlines():
    if "KV cache" in line or "concurrency" in line:
        print("  ", line[-160:])

/usr/local/bin/vllm serve /kaggle/input/notebooks/redwannewaz/t2-prep-model/model --served-model-name disasterm3-7b --host 127.0.0.1 --port 8000 --tensor-parallel-size 2 --dtype half --enforce-eager --max-model-len 32768 --gpu-memory-utilization 0.92 --max-num-seqs 8 --limit-mm-per-prompt {"image": 1} --mm-processor-kwargs {"max_pixels": 1806336}
[  7.8 min] vLLM is ready
   (Worker_TP0 pid=140) INFO 11-20 02:21:29 [gpu_worker.py:538] Available KV cache memory: 5.29 GiB
   (EngineCore pid=114) INFO 11-20 02:21:29 [kv_cache_utils.py:2146] GPU KV cache size: 198,176 tokens
   (EngineCore pid=114) INFO 11-20 02:21:29 [kv_cache_utils.py:2147] Maximum concurrency for 32,768 tokens per request: 6.05x


Four short tests through `local_model.py`, the same code DisasTeller uses: text, one picture, a call through CrewAI, and a question as the fixed flow asks it (`ask_text`).

In [9]:
os.environ.update(LOCAL_LLM_PROVIDER="hosted_vllm", LOCAL_LLM_MODEL=SERVED_NAME,
                  LOCAL_LLM_BASE_URL=f"http://127.0.0.1:{PORT}/v1", LOCAL_LLM_CONTEXT=str(MAX_MODEL_LEN),
                  CREWAI_DISABLE_TELEMETRY="true", CREWAI_TRACING_ENABLED="false",
                  CREWAI_DISABLE_VERSION_CHECK="true", LOCAL_LLM_CALL_LOG=f"{WORK}/smoke_llm_calls.jsonl")
sys.path.insert(0, DT_DIR)
import base64, io
from PIL import Image
import local_model

served = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models"))
print("Server has:", [m["id"] for m in served["data"]])

buf = io.BytesIO()
Image.open(f"{DT_DIR}/Disaster_image/Local_image/Picture1.png").convert("RGB").resize((720, 720)).save(buf, format="PNG")
photo = base64.b64encode(buf.getvalue()).decode()

SMOKE = {}
for label, ask in [
    ("text", lambda: local_model.ask_with_images("In one sentence: what is a building damage grade?", max_tokens=80)),
    ("picture", lambda: local_model.ask_with_images("Describe the damage in this image in two sentences.", [photo], max_tokens=150)),
    ("crewai", lambda: str(local_model.crew_llm().call([{"role": "user", "content": "Reply with the single word OK."}]))),
    ("fixed flow", lambda: local_model.ask_text("You are a test.", "Reply with the single word OK.", max_tokens=10)),
]:
    t = time.time()
    SMOKE[label] = answer = ask()
    print(f"{label} ({time.time() - t:.0f} s): {answer[:300]}")
    if answer.startswith("An error occurred"):
        raise RuntimeError(f"The {label} test failed. See t2_run/vllm_server.log.")

Server has: ['disasterm3-7b']
text (6 s): A building damage grade refers to the level of structural damage sustained by a building as a result of an event, such as an earthquake or hurricane.
picture (3 s): The Wajima Drama Memorial Hall has sustained no visible damage as a result of the disaster.
crewai (0 s): OK
fixed flow (0 s): OK


**The run.** `run_with_netlog.py` runs DisasTeller (the `FLOW` of the first cell) and records every
network connection it makes. Settings: offline mode, the Kaggle search backend (`GUIDELINE_BACKEND=numpy`; CrewAI brings chromadb, which
would otherwise pick the Chroma way and try to download its model), and no GPU for DisasTeller itself
(the search model runs on the CPU; the GPUs belong to vLLM). Only key lines are printed here;
everything is in `t2_run/run/run_log.txt`.

In [10]:
LOCAL = os.path.join(DT_DIR, "Disaster_image", "Local_image")
KEY = ("Local model:", "DisasTeller mode", "DisasTeller flow", "Guideline index loaded", "Local information (",
       "Labelled map saved", "Local model call failed", "Local model gave no answer", "Photo tool asked",
       "Model message shortened", "Section written", "Section asked again", "Section kept", "Grades found",
       "Report finished", "Fixed flow finished", "Network log", "Outside this computer", "Gemini libraries",
       "Main script finished", "Traceback", "No C4 facts", "Set C4_SCENE", "must hold only")
RESULTS = {}
for scene, info in SCENES.items():
    # The photo tool describes every picture in Local_image/: only this pair, before first
    shutil.rmtree(LOCAL, ignore_errors=True)
    os.makedirs(LOCAL)
    shutil.copy2(PHOTOS[(scene, "before")], os.path.join(LOCAL, f"1_before_{info['before_image']}"))
    shutil.copy2(PHOTOS[(scene, "after")], os.path.join(LOCAL, f"2_after_{scene}"))
    RUN_DIR = f"{WORK}/run_{os.path.splitext(scene)[0]}"
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    os.makedirs(RUN_DIR)
    env = {k: v for k, v in os.environ.items() if k != "DISASTELLER_ONLINE"}
    env.update(LOCAL_LLM_PROVIDER="hosted_vllm", LOCAL_LLM_MODEL=SERVED_NAME,
               LOCAL_LLM_BASE_URL=f"http://127.0.0.1:{PORT}/v1", LOCAL_LLM_CONTEXT=str(MAX_MODEL_LEN),
               GUIDELINE_BACKEND="numpy", GUIDELINE_INDEX_DIR=RAG_DIR, HF_HUB_OFFLINE="1",
               TRANSFORMERS_OFFLINE="1", CUDA_VISIBLE_DEVICES="", PYTHONDONTWRITEBYTECODE="1",
               PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1", DISASTELLER_FLOW=FLOW, C4_SCENE=scene,
               LOCAL_LLM_CALL_LOG="llm_calls.jsonl")  # in RUN_DIR
    print(f"\n[{minutes()}] ===== {scene} ({info['disaster']}, rows: {info['kind']})")
    proc = subprocess.Popen([sys.executable, os.path.join(DT_DIR, "run_with_netlog.py")], cwd=RUN_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    timer = threading.Timer(RUN_TIMEOUT_HOURS * 3600, proc.kill)
    timer.start()
    for line in proc.stdout:
        if any(k in line for k in KEY):
            print(f"[{minutes()}] {line.rstrip()[:220]}")
    proc.wait()
    timer.cancel()
    RESULTS[scene] = {"run_dir": RUN_DIR, "flow_exit_code": proc.returncode}
    print(f"[{minutes()}] {scene}: flow exit code {proc.returncode}")


[  8.0 min] ===== turkey_earthquake4_post_327.png (earthquake, rows: outline)
[  8.1 min] Local model: disasterm3-7b at http://127.0.0.1:8000/v1 (hosted_vllm)
[  8.1 min] DisasTeller mode: offline
[  8.1 min] DisasTeller flow: fixed, with C4's damage facts (scene turkey_earthquake4_post_327.png, earthquake, rows: outline)
[  8.1 min] Photo tool asked: describe the photos
[  8.1 min] Local information (1_before_turkey_earthquake4_pre_327.png): describe the disaster in detail in this image, in about 200 words. If the image has a sticker or sign with a place name, start with that place name. Describe o
[  8.2 min] Local information (2_after_turkey_earthquake4_post_327.png): describe the disaster in detail in this image, in about 200 words. If the image has a sticker or sign with a place name, start with that place name. Describe o
[  8.5 min] Guideline index loaded in 11.3 s (numpy + torch), once per run
[  8.8 min] Section written: Expert_team_report.txt / Disaster grade of each locatio

**The pass rules** (fixed in the README before any run), checked by `check_run.py`.

In [11]:
for scene, result in RESULTS.items():
    check = subprocess.run([sys.executable, os.path.join(DT_DIR, "check_run.py"), result["run_dir"], RAG_DIR,
                            "--internet", INTERNET], capture_output=True, text=True)
    result["check_exit_code"] = check.returncode
    print(f"\n===== {scene}: the pass rules (exit code {check.returncode})")
    print(check.stdout, check.stderr[-2000:])
    path = f"{result['run_dir']}/reports/offline/Expert_team_report.txt"
    if os.path.isfile(path):
        print("----- Expert_team_report.txt (first 2500 characters) -----")
        print(open(path, encoding="utf-8", errors="replace").read()[:2500])


===== turkey_earthquake4_post_327.png: the pass rules (exit code 0)
PASS  1. The run finished
PASS  2. All six reports, each starting 'Mode: offline', each 50+ words
PASS  3. Nothing went outside this computer
PASS  4. No Gemini library was loaded
PASS  5. Every model question got an answer

PASSED: the offline proof (internet off)

report                                             words (without headings)  cites(real)  'not available offline'
Expert_team_report.txt                               269 (  238)               5 (  5)    0
Alerts_team_report_for_social_media.txt              201 (  182)               0 (  0)    0
Emergency_team_report.txt                            317 (  298)               0 (  0)    0
Emergency_service_human_resource_allocation.txt      651 (  623)               6 (  6)    0
Public_community_report.txt                          554 (  534)               9 (  9)    0
Reconstruction_plan.txt                              754 (  732)              17 ( 17)    

In [12]:
server.terminate()
try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    server.kill()

info = {
    "internet": INTERNET,
    "flow": FLOW,
    "scenes": RESULTS,
    "minutes": round((time.time() - T0) / 60, 1),
    "served_name": SERVED_NAME,
    "max_model_len": MAX_MODEL_LEN,
    "versions": VERSIONS,
    "model": model_source["repo"],
    "model_commit": model_source["commit"],
    "code_packed": code["made"],
    "code_git_commit": code["git_commit"],
    "code_uncommitted_changes": code["uncommitted_changes_in_disasteller_offline"],
    "code_files_found_at": FOUND_AT,
    "smoke_tests": SMOKE,
}
with open(f"{WORK}/t2_notebook_info.json", "w") as f:
    json.dump(info, f, indent=2)
zip_path = shutil.make_archive(f"/kaggle/working/t2_step5_c4_internet_{INTERNET}", "zip", WORK)
print(json.dumps({k: v for k, v in info.items() if k != "smoke_tests"}, indent=2))
print("Zip:", zip_path)

{
  "internet": "off",
  "flow": "c4",
  "scenes": {
    "turkey_earthquake4_post_327.png": {
      "run_dir": "/kaggle/working/t2_step5/run_turkey_earthquake4_post_327",
      "flow_exit_code": 0,
      "check_exit_code": 0
    },
    "santa_rosa_wildfire_00000155_post_disaster.png": {
      "run_dir": "/kaggle/working/t2_step5/run_santa_rosa_wildfire_00000155_post_disaster",
      "flow_exit_code": 0,
      "check_exit_code": 0
    }
  },
  "minutes": 19.1,
  "served_name": "disasterm3-7b",
  "max_model_len": 32768,
  "versions": {
    "vllm": "0.25.1",
    "crewai": "1.15.2",
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "openai": "2.54.0",
    "numpy": "2.1.3",
    "pillow": "13.0.0"
  },
  "model": "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP",
  "model_commit": "def8f6c81889ffb68904afdc90c3a1389697d73b",
  "code_packed": "2026-11-20 07:57:14",
  "code_git_commit": "1a5f38094",
  "code_uncommitted_changes": true,
  "code_files_found_at": {
    "DisasterManageme

**After the run:** download `t2_step5_c4_internet_off.zip` from the Output tab, unzip it into
`Offline_RAG_KR2/results/t2_step5_c4/`, and save the run notebook over `t2_step5_c4_run.ipynb`. Never change an
output by hand. Step 7 then checks every number in the twelve reports against the table.